# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from build_graph_and_train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 200

In [3]:
df = pd.read_csv(f"../../../data/top30groups/noGeographic/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Weapon type prediction

In [6]:
torch.cuda.empty_cache()


In [7]:
import random


label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds, y_trues, logs = [], [], []
from itertools import product
import os
# Hyperparameter grid
param_grid = {
    'lr': [0.001, 0.01],
    'n_tree': [40, 80, 100],
    'tree_depth': [8, 10],
    'tree_feature_rate': [0.1, 0.3, 0.5],
    'feat_dropout': [0.0, 0.1, 0.2],
    'embed_dim': [16, 32, 64]
    }

# Convert to list of dicts (cartesian product), then randomly sample 10
all_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

# Sample 10 random configurations from the full grid
random.seed(42)  # for reproducibility
grid_combos = random.sample(all_combos, k=100)

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")

    data, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data_knn(
        df, label_index, continuous_col=col)

    best_run = None
    best_score = -1
    best_args = None

    # -----------------------------
    # STAGE 1: Hyperparameter search
    # -----------------------------
    for combo in grid_combos:
        args = {
            **dict(zip(param_names, combo)),
            'partition': f"gtd{partition}",
            'n_class': len(label_index),
            'epochs': 1500,
            'final_evaluation': False  # <-- Only validate
        }

        print(f"[SEARCH] Trying config: {args}")
        
        acc, epoch, *_ = train_joint(
            data, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask,
            args, row_to_node_index, index_to_label, verbose=False)

        if acc > best_score:
            best_score = acc
            best_args = args.copy()


    # -----------------------------
    # STAGE 2: Final evaluation
    # -----------------------------
    if best_args:
        print(f"\n[FINAL] Retraining with best config from search: {best_args}")
        best_args['epochs'] = 3000
        best_args['final_evaluation'] = True

        try:
            acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(
                data, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask,
                best_args, row_to_node_index, index_to_label, verbose=False)

            best_run = {
                "args": best_args,
                "acc": acc,
                "epoch": epoch,
                "y_pred": y_pred_decoded,
                "y_true": y_true_decoded,
                "precision": p,
                "recall": r,
                "f1": f1,
                "micro": (p_micro, r_micro, f1_micro),
                "macro": (p_macro, r_macro, f1_macro),
                "auroc": (auc_w, auc_mi, auc_ma),
                "epoch_logs": epoch_logs
            }

        except Exception as e:
            print(f"Error during final evaluation with config {best_args}: {e}")

    # -----------------------------
    # SAVE RESULTS
    # -----------------------------
    if best_run:
        args = best_run["args"]
        os.makedirs(f"Results{partition}", exist_ok=True)

        results_path = f"Results{partition}/Results_{col}_prediction"
        with open(results_path, "w") as f:
            f.write(f"Best acc: {best_run['acc']:.4f} at epoch {best_run['epoch']} for {col} prediction\n")
            f.write(f"Config: {args}\n")
            f.write(f"Weighted Precision: {best_run['precision']:.4f}, Recall: {best_run['recall']:.4f}, F1: {best_run['f1']:.4f}\n")
            f.write(f"Macro Precision: {best_run['macro'][0]:.4f}, Recall: {best_run['macro'][1]:.4f}, F1: {best_run['macro'][2]:.4f}\n")
            f.write(f"Micro Precision: {best_run['micro'][0]:.4f}, Recall: {best_run['micro'][1]:.4f}, F1: {best_run['micro'][2]:.4f}\n")
            f.write(f"AUROC Weighted: {best_run['auroc'][0]:.4f}, Micro: {best_run['auroc'][1]:.4f}, Macro: {best_run['auroc'][2]:.4f}\n")

        log_path = f"Results{partition}/epoch_logs_{col}_prediction"
        with open(log_path, "w") as f:
            f.write('\n'.join(f"{x:.4f}" for x in best_run['epoch_logs']))

        y_preds.append(best_run['y_pred'])
        y_trues.append(best_run['y_true'])

print(best_score)



Training model for weaptype1 prediction...
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 16, 'partition': 'gtd200', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 960
Best acc/epoch: 0.5472779273986816, epoch 860
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 16, 'partition': 'gtd200', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 1170
Best acc/epoch: 0.5859599113464355, epoch 1070
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd200', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 1266
Best acc/epoch: 0.5859599113464355, epoch 1166
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 100, 'tree_depth': 8, 'tree_feature_ra

In [8]:
"""

Training model for weaptype1 prediction...
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 842
Best acc/epoch: 0.4801980257034302, epoch 742
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 630
Best acc/epoch: 0.44306930899620056, epoch 530
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 842
Best acc/epoch: 0.48762375116348267, epoch 742
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 510
Best acc/epoch: 0.4529702961444855, epoch 410
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 614
Best acc/epoch: 0.45049503445625305, epoch 514
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 601
Best acc/epoch: 0.4727722704410553, epoch 501
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 899
Best acc/epoch: 0.4653465151786804, epoch 799
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 758
Best acc/epoch: 0.4653465151786804, epoch 658
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 590
Best acc/epoch: 0.4801980257034302, epoch 490
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 560
Best acc/epoch: 0.4455445408821106, epoch 460
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 611
Best acc/epoch: 0.4628712832927704, epoch 511
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 1080
Best acc/epoch: 0.4727722704410553, epoch 980
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 676
Best acc/epoch: 0.44059404730796814, epoch 576
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 1087
Best acc/epoch: 0.48762375116348267, epoch 987
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 480
Best acc/epoch: 0.4826732575893402, epoch 380
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 392
Best acc/epoch: 0.4702970087528229, epoch 292
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 495
Best acc/epoch: 0.4653465151786804, epoch 395
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 350
Best acc/epoch: 0.48514851927757263, epoch 250
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 269
Best acc/epoch: 0.4628712832927704, epoch 169
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 269
Best acc/epoch: 0.44306930899620056, epoch 169
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 926
Best acc/epoch: 0.4900990128517151, epoch 826
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 619
Best acc/epoch: 0.46039602160453796, epoch 519
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 247
Best acc/epoch: 0.4727722704410553, epoch 147
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 291
Best acc/epoch: 0.4628712832927704, epoch 191
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 277
Best acc/epoch: 0.4628712832927704, epoch 177
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 323
Best acc/epoch: 0.47772276401519775, epoch 223
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 286
Best acc/epoch: 0.45792078971862793, epoch 186
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 412
Best acc/epoch: 0.4727722704410553, epoch 312
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 663
Best acc/epoch: 0.44306930899620056, epoch 563
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 245
Best acc/epoch: 0.4529702961444855, epoch 145
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 461
Best acc/epoch: 0.44059404730796814, epoch 361
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 1128
Best acc/epoch: 0.47772276401519775, epoch 1028
[SEARCH] Trying config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 239
Best acc/epoch: 0.4554455280303955, epoch 139
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early stopping at epoch 658
Best acc/epoch: 0.448019802570343, epoch 558
[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
"""

"\n\nTraining model for weaptype1 prediction...\n[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}\nEarly stopping at epoch 842\nBest acc/epoch: 0.4801980257034302, epoch 742\n[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}\nEarly stopping at epoch 630\nBest acc/epoch: 0.44306930899620056, epoch 530\n[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}\nEarly stopping at epoch 842\nBest acc/epoch: 0.48762375116348267, epoch 742\n[SEARCH] Trying config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'tre

In [9]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [10]:
print(best_run)

{'args': {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd200', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.5290610790252686, 'epoch': 1351, 'y_pred': ['Fulani extremists', 'Al-Qaida in the Arabian Peninsula (AQAP)', "Kurdistan Workers' Party (PKK)", 'African National Congress (South Africa)', 'African National Congress (South Africa)', 'Farabundo Marti National Liberation Front (FMLN)', 'Taliban', 'Islamic State of Iraq and the Levant (ISIL)', 'Al-Qaida in the Arabian Peninsula (AQAP)', 'Palestinians', 'African National Congress (South Africa)', 'Sikh Extremists', 'Palestinians', "New People's Army (NPA)", 'National Liberation Army of Colombia (ELN)', 'Al-Qaida in the Arabian Peninsula (AQAP)', 'Boko Haram', 'Shining Path (SL)', 'Tehrik-i-Taliban Pakistan (TTP)', 'Tupac Amaru Revolutionary Movement (MRTA)', 'Irish Republican Army (IRA)', 'African National Congress (South Africa)', 'Pal

In [11]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1000,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': len(label_index),\n    \'final_evaluation\': True\n}\n0.9287652969360352\n\n'

In [12]:
best_acc

NameError: name 'best_acc' is not defined

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

'\nBest acc: 0.9205 at epoch 750 for weaptype1 prediction\nWeighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191\nMacro Precision: 0.9176, Recall: 0.9107, F1: 0.9105\nMicro Precision: 0.9205, Recall: 0.9205, F1: 0.9205\nAUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963\n\n'

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.98      0.93      0.95        43
        African National Congress (South Africa)       0.98      1.00      0.99        58
                                Al-Qaida in Iraq       1.00      0.79      0.88        87
        Al-Qaida in the Arabian Peninsula (AQAP)       0.98      0.92      0.95        50
                                      Al-Shabaab       1.00      1.00      1.00        58
             Basque Fatherland and Freedom (ETA)       0.98      0.90      0.94        60
                                      Boko Haram       0.93      0.95      0.94        41
  Communist Party of India - Maoist (CPI-Maoist)       1.00      0.92      0.96        24
       Corsican National Liberation Front (FLNC)       0.97      0.96      0.97        78
                       Donetsk People's Republic       1.00      1.00      1.00        59
Farabundo

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])

Saved confusion matrix for partition 100 to Results100/cm_100_weaptype1.png
